In [23]:
import pandas as pd
import numpy as np
import subprocess
import sys
import mlflow

In [24]:
# Carregando o dataset tratado para o repositório
df = pd.read_parquet('../data/processed/dataset_tratado.parquet')

In [25]:
import mlflow
import mlflow.sklearn

# Configurar MLflow
mlflow.set_experiment("churn_prediction_models")
print("✓ MLflow configurado!")
print(f"Experiment: {mlflow.get_experiment_by_name('churn_prediction_models').name}")


✓ MLflow configurado!
Experiment: churn_prediction_models


# --------------- REGRESSÃO LOGISTICA ---------------

In [26]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

#separando variveis alvo e features
X = df.drop(columns=['churn_value'])
y = df['churn_value']


# split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


#identificando colunas categóricas e numéricas
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X_train.select_dtypes(include='object').columns


#criando o one hot encoder para as colunas categóricas
encoder = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)

#criando o column transformer para aplicar o one hot encoder apenas nas colunas categóricaspreprocessador = ColumnTransformer(
preprocessador = ColumnTransformer(
    transformers=[
        ('cat', encoder, categorical_cols),
        ('num', StandardScaler(), numerical_cols) #normalizar as colunas numéricas
    ]
)

#aplicando transformação
X_train_transformado = preprocessador.fit_transform(X_train)

X_test_transformado = preprocessador.transform(X_test)

#transformando o resultado em um dataframe e nomeando as colunas
feature_names = preprocessador.get_feature_names_out()

# garantir que o retorno seja array (não spmatrix nos casos de sparse_output variável)
if hasattr(X_train_transformado, "toarray"):
    X_train_transformado = X_train_transformado.toarray()
if hasattr(X_test_transformado, "toarray"):
    X_test_transformado = X_test_transformado.toarray()

X_train_transformado = pd.DataFrame(
    X_train_transformado,
    columns=feature_names
)

X_test_transformado = pd.DataFrame(
    X_test_transformado,
    columns=feature_names
)

X_test_transformado = pd.DataFrame(
    X_test_transformado,
    columns=feature_names
)

# exibindo as shapes dos dataframes transformadoss
print("Shape treino:", X_train_transformado.shape)
print("Shape teste:", X_test_transformado.shape)



Shape treino: (5634, 2845)
Shape teste: (1409, 2845)


In [27]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix, classification_report

with mlflow.start_run(run_name="logistic_regression"):
    model = LogisticRegression(random_state=42, max_iter=1000)

    mlflow.log_params({
        "model_type": "LogisticRegression",
        "max_iter": 1000,
        "random_state": 42
    })

    model.fit(X_train_transformado, y_train)

    y_pred_train = model.predict(X_train_transformado)
    y_pred_test = model.predict(X_test_transformado)

    acc = accuracy_score(y_test, y_pred_test)
    prec = precision_score(y_test, y_pred_test)
    rec = recall_score(y_test, y_pred_test)
    f1 = f1_score(y_test, y_pred_test)
    roc_auc = roc_auc_score(y_test, model.predict_proba(X_test_transformado)[:, 1])
    cm = confusion_matrix(y_test, y_pred_test)

    mlflow.log_metrics({
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1_score": f1,
        "roc_auc": roc_auc
    })

    mlflow.sklearn.log_model(model, "logistic_regression_model")

    print("Logistic Regression - Métricas de validação")
    print(f"Accuracy: {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall: {rec:.4f}")
    print(f"F1-score: {f1:.4f}")
    print(f"ROC AUC: {roc_auc:.4f}")
    print("Confusion Matrix:\n", cm)
    print("Classification Report:\n", classification_report(y_test, y_pred_test))

2026/03/28 16:36:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/28 16:36:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logistic Regression - Métricas de validação
Accuracy: 0.9681
Precision: 0.9970
Recall: 0.8824
F1-score: 0.9362
ROC AUC: 0.9798
Confusion Matrix:
 [[1034    1]
 [  44  330]]
Classification Report:
               precision    recall  f1-score   support

           0       0.96      1.00      0.98      1035
           1       1.00      0.88      0.94       374

    accuracy                           0.97      1409
   macro avg       0.98      0.94      0.96      1409
weighted avg       0.97      0.97      0.97      1409



In [28]:
# A avaliação da regressão logística já está feita no bloco de treino principal
# Este bloco é mantido para compatibilidade, mas não reexecuta o mesmo run.

print("Logistic Regression model already trained and métricas logadas no MLflow no bloco anterior.")


Logistic Regression model already trained and métricas logadas no MLflow no bloco anterior.


# --------------- DUMMY CLASSIFIER ---------------

In [29]:
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline

# =========================
# Separando features e alvo
# =========================
X = df.drop(columns=['churn_value'])
y = df['churn_value']

# =========================
# Split treino e teste
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# =========================
# Identificando colunas
# =========================
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X_train.select_dtypes(include='object').columns

# =========================
# One Hot Encoder
# =========================
encoder = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)

# =========================
# Preprocessador
# =========================
preprocessador = ColumnTransformer(
    transformers=[
        ('cat', encoder, categorical_cols),
        ('num', StandardScaler(), numerical_cols)
    ]
)

# =========================
# Seed
# =========================
SEED = 20
np.random.seed(SEED)

# =========================
# Dummy Classifier
# =========================
modelo_dummy = DummyClassifier(
    strategy='stratified',
    random_state=SEED
)

# =========================
# Pipeline (boa prática)
# =========================
pipeline_dummy = Pipeline([
    ('preprocessamento', preprocessador),
    ('modelo', modelo_dummy)
])

# =========================
# Validação cruzada
# =========================
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=SEED
)

resultados = cross_validate(
    pipeline_dummy,
    X_train,
    y_train,
    cv=cv,
    scoring=['accuracy', 'precision', 'recall', 'f1'],
    return_train_score=True
)

# =========================
# Resultados
# =========================
print("===== Dummy Classifier =====")
print("Accuracy média:", resultados['test_accuracy'].mean())
print("F1 média:", resultados['test_f1'].mean())
print("Precision média:", resultados['test_precision'].mean())
print("Recall média:", resultados['test_recall'].mean())

===== Dummy Classifier =====
Accuracy média: 0.6006348966403385
F1 média: 0.2704135670155087
Precision média: 0.2624432768091713
Recall média: 0.2788859060402684


# --------------- REGRESSÃO LINEAR ---------------

In [30]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np

with mlflow.start_run(run_name="linear_regression"):
    lr_model = LinearRegression()
    mlflow.log_param("model_type", "LinearRegression")

    lr_model.fit(X_train_transformado, y_train)

    y_pred_train_lr = lr_model.predict(X_train_transformado)
    y_pred_test_lr = lr_model.predict(X_test_transformado)

    mse_train = mean_squared_error(y_train, y_pred_train_lr)
    mse_test = mean_squared_error(y_test, y_pred_test_lr)
    rmse_train = np.sqrt(mse_train)
    rmse_test = np.sqrt(mse_test)
    mae_train = mean_absolute_error(y_train, y_pred_train_lr)
    mae_test = mean_absolute_error(y_test, y_pred_test_lr)
    r2_train = r2_score(y_train, y_pred_train_lr)
    r2_test = r2_score(y_test, y_pred_test_lr)

    mlflow.log_metrics({
        "mse_train": mse_train,
        "mse_test": mse_test,
        "rmse_train": rmse_train,
        "rmse_test": rmse_test,
        "mae_train": mae_train,
        "mae_test": mae_test,
        "r2_train": r2_train,
        "r2_test": r2_test
    })

    mlflow.sklearn.log_model(lr_model, "linear_regression_model")

    print("Linear Regression - Métricas")
    print(f"RMSE Treino: {rmse_train:.4f}")
    print(f"RMSE Teste: {rmse_test:.4f}")
    print(f"MSE Treino: {mse_train:.4f}")
    print(f"MSE Teste: {mse_test:.4f}")
    print(f"MAE Treino: {mae_train:.4f}")
    print(f"MAE Teste: {mae_test:.4f}")
    print(f"R² Treino: {r2_train:.4f}")
    print(f"R² Teste: {r2_test:.4f}")

2026/03/28 16:37:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/28 16:37:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Linear Regression - Métricas
RMSE Treino: 0.1317
RMSE Teste: 0.1911
MSE Treino: 0.0173
MSE Teste: 0.0365
MAE Treino: 0.0587
MAE Teste: 0.0866
R² Treino: 0.9110
R² Teste: 0.8127


# --------------- ÁRVORE DE DECISÃO ---------------

In [31]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

with mlflow.start_run(run_name="decision_tree"):
    modelo_tree = DecisionTreeClassifier(max_depth=5, random_state=42)
    mlflow.log_params({
        "model_type": "DecisionTreeClassifier",
        "max_depth": 5,
        "random_state": 42
    })

    modelo_tree.fit(X_train_transformado, y_train)

    y_pred_train_tree = modelo_tree.predict(X_train_transformado)
    y_pred_test_tree = modelo_tree.predict(X_test_transformado)

    acc_train_tree = accuracy_score(y_train, y_pred_train_tree)
    acc_test_tree = accuracy_score(y_test, y_pred_test_tree)
    prec_tree = precision_score(y_test, y_pred_test_tree)
    rec_tree = recall_score(y_test, y_pred_test_tree)
    f1_tree = f1_score(y_test, y_pred_test_tree)
    roc_auc_tree = roc_auc_score(y_test, modelo_tree.predict_proba(X_test_transformado)[:, 1])
    cm_tree = confusion_matrix(y_test, y_pred_test_tree)
    overfitting_tree = acc_train_tree - acc_test_tree

    mlflow.log_metrics({
        "accuracy_train": acc_train_tree,
        "accuracy_test": acc_test_tree,
        "precision": prec_tree,
        "recall": rec_tree,
        "f1_score": f1_tree,
        "roc_auc": roc_auc_tree,
        "overfitting": overfitting_tree
    })

    mlflow.sklearn.log_model(modelo_tree, "decision_tree_model")

    print("Decision Tree - Métricas")
    print(f"Accuracy Treino: {acc_train_tree:.4f}")
    print(f"Accuracy Teste: {acc_test_tree:.4f}")
    print(f"Precision: {prec_tree:.4f}")
    print(f"Recall: {rec_tree:.4f}")
    print(f"F1-score: {f1_tree:.4f}")
    print(f"ROC AUC: {roc_auc_tree:.4f}")
    print(f"Overfitting: {overfitting_tree:.4f}")
    print("Confusion Matrix:\n", cm_tree)
    print("Classification Report:\n", classification_report(y_test, y_pred_test_tree))


2026/03/28 16:37:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/28 16:37:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Decision Tree - Métricas
Accuracy Treino: 0.9759
Accuracy Teste: 0.9674
Precision: 0.9824
Recall: 0.8930
F1-score: 0.9356
ROC AUC: 0.9768
Overfitting: 0.0085
Confusion Matrix:
 [[1029    6]
 [  40  334]]
Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.99      0.98      1035
           1       0.98      0.89      0.94       374

    accuracy                           0.97      1409
   macro avg       0.97      0.94      0.96      1409
weighted avg       0.97      0.97      0.97      1409



# --------------- RANDOM FOREST ---------------

In [32]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix, classification_report

with mlflow.start_run(run_name="random_forest"):
    rf_model = RandomForestClassifier(random_state=42, n_estimators=100)
    mlflow.log_params({
        "model_type": "RandomForestClassifier",
        "n_estimators": 100,
        "random_state": 42
    })

    rf_model.fit(X_train_transformado, y_train)

    y_pred_train_rf = rf_model.predict(X_train_transformado)
    y_pred_test_rf = rf_model.predict(X_test_transformado)

    acc_train_rf = accuracy_score(y_train, y_pred_train_rf)
    acc_test_rf = accuracy_score(y_test, y_pred_test_rf)
    prec_rf = precision_score(y_test, y_pred_test_rf)
    rec_rf = recall_score(y_test, y_pred_test_rf)
    f1_rf = f1_score(y_test, y_pred_test_rf)
    roc_auc_rf = roc_auc_score(y_test, rf_model.predict_proba(X_test_transformado)[:, 1])
    cm_rf = confusion_matrix(y_test, y_pred_test_rf)
    overfitting_rf = acc_train_rf - acc_test_rf

    mlflow.log_metrics({
        "accuracy_train": acc_train_rf,
        "accuracy_test": acc_test_rf,
        "precision": prec_rf,
        "recall": rec_rf,
        "f1_score": f1_rf,
        "roc_auc": roc_auc_rf,
        "overfitting": overfitting_rf
    })

    mlflow.sklearn.log_model(rf_model, "random_forest_model")

    print("Random Forest - Métricas")
    print(f"Accuracy Treino: {acc_train_rf:.4f}")
    print(f"Accuracy Teste: {acc_test_rf:.4f}")
    print(f"Precision: {prec_rf:.4f}")
    print(f"Recall: {rec_rf:.4f}")
    print(f"F1-score: {f1_rf:.4f}")
    print(f"ROC AUC: {roc_auc_rf:.4f}")
    print(f"Overfitting: {overfitting_rf:.4f}")
    print("Confusion Matrix:\n", cm_rf)
    print("Classification Report:\n", classification_report(y_test, y_pred_test_rf))


2026/03/28 16:37:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/28 16:37:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Random Forest - Métricas
Accuracy Treino: 1.0000
Accuracy Teste: 0.9674
Precision: 0.9881
Recall: 0.8877
F1-score: 0.9352
ROC AUC: 0.9771
Overfitting: 0.0326
Confusion Matrix:
 [[1031    4]
 [  42  332]]
Classification Report:
               precision    recall  f1-score   support

           0       0.96      1.00      0.98      1035
           1       0.99      0.89      0.94       374

    accuracy                           0.97      1409
   macro avg       0.97      0.94      0.96      1409
weighted avg       0.97      0.97      0.97      1409



In [38]:
# Resumo de todos os experimentos MLflow
experiment = mlflow.get_experiment_by_name('churn_prediction_models')

if experiment is None:
    print("Experimento 'churn_prediction_models' não encontrado.")
    print("Verifique se o experimento foi criado antes de buscar runs.")
else:
    runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
    print("RESUMO DE TODOS OS MODELOS TREINADOS")
    print("="*80)

    if runs.empty:
        print("Nenhuma run encontrada para o experimento.")
    else:
        for idx, run in runs.iterrows():
            print(f"\nRun: {run['tags.mlflow.runName']}")
            print(f"Status: {run['status']}")
            print("Métricas:")
            for col in runs.columns:
                if col.startswith('metrics.'):
                    metric_name = col.replace('metrics.', '')
                    metric_value = run[col]
                    if pd.notna(metric_value):
                        print(f"  {metric_name}: {metric_value:.4f}")


RESUMO DE TODOS OS MODELOS TREINADOS

Run: random_forest
Status: FINISHED
Métricas:
  recall: 0.8877
  accuracy_test: 0.9674
  overfitting: 0.0326
  accuracy_train: 1.0000
  roc_auc: 0.9771
  f1_score: 0.9352
  precision: 0.9881

Run: decision_tree
Status: FINISHED
Métricas:
  recall: 0.8930
  accuracy_test: 0.9674
  overfitting: 0.0085
  accuracy_train: 0.9759
  roc_auc: 0.9768
  f1_score: 0.9356
  precision: 0.9824

Run: linear_regression
Status: FINISHED
Métricas:
  rmse_train: 0.1317
  mae_test: 0.0866
  r2_test: 0.8127
  mse_test: 0.0365
  mse_train: 0.0173
  r2_train: 0.9110
  mae_train: 0.0587
  rmse_test: 0.1911

Run: logistic_regression
Status: FINISHED
Métricas:
  recall: 0.8824
  roc_auc: 0.9798
  f1_score: 0.9362
  precision: 0.9970
  accuracy: 0.9681

Run: logistic_regression
Status: FINISHED
Métricas:
  recall: 0.8824
  roc_auc: 0.9798
  f1_score: 0.9362
  precision: 0.9970
  accuracy: 0.9681
